# iMatch R1 지도 특징 재사용 실험
Kaggle에서 GPU와 Internet을 켜고 `heeyunju/visloc-seq-eval`을 추가한 뒤 Run All을 실행하세요. 두 평가 셀은 같은 LoMa-B와 4방향(0°, 90°, 180°, 270°), inlier 15개 조기 종료, VIO seed 2026을 사용합니다. 한 걸음 안에서 두 번째 방향부터 같은 지도 특징을 재사용합니다. 주 지표는 92구간 전체 재생 시간입니다. 시작점 포함 1,112프레임의 추정 위치와 판정 차이를 마지막 셀에서 확인하세요.

In [ ]:
from pathlib import Path
import sys, subprocess, json, datetime, shutil, hashlib, os
import pandas as pd
import numpy as np
import cv2
import torch
from IPython.display import FileLink, display
if not torch.cuda.is_available(): raise RuntimeError('Kaggle에서 GPU를 켜 주세요.')
WORK=Path('/kaggle/working/imatch_r1_map_cache_4dir')
RUN=WORK/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE=RUN/'source'
SOURCE.mkdir(parents=True)
INPUT_ROOT=Path('/kaggle/input')
DATA_DIR=''
SETS=[]
SEED=2026
ODO='vio'
def save_json(path,value):
    path.write_text(json.dumps(value,indent=2,default=str),encoding='utf8')
def hash_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
print(torch.cuda.get_device_name(), RUN)

In [ ]:
import zipfile
required={'set','k','map_x','map_y','dist_m'}
def candidates(root):
    found=[]
    for p in Path(root).rglob('samples.csv'):
        if required.issubset(pd.read_csv(p,nrows=0).columns): found.append(p.parent)
    return sorted(set(found))
search=Path(DATA_DIR) if DATA_DIR else INPUT_ROOT
roots=candidates(search)
# seq_eval.zip 또는 samples.csv + maps.zip/구간별 ZIP 구조를 모두 지원
archives=list(search.rglob('*.zip'))
if archives and not any((r/'maps').is_dir() and next(r.glob('*/k*_uav.jpg'),None) for r in roots):
    extracted=RUN.parent/('input_'+RUN.name); extracted.mkdir(exist_ok=True)
    relevant=[]
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            names=z.namelist()
            if any(Path(n).name=='samples.csv' or '/maps/' in '/'+n or Path(n).name.startswith('k') and n.endswith('_uav.jpg') for n in names):
                relevant.append(archive)
    for archive in relevant:
        target=extracted/str(archive.parent.relative_to(search))
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                if not (target/member.filename).resolve().is_relative_to(target.resolve()): raise ValueError('Unsafe ZIP path')
            z.extractall(target)
        for loose in archive.parent.iterdir():
            if loose.is_file() and loose.suffix=='.csv': shutil.copy2(loose,target/loose.name)
            elif loose.is_dir(): shutil.copytree(loose,target/loose.name,dirs_exist_ok=True)
    roots=candidates(extracted)
if len(roots)!=1: raise ValueError(f'Choose DATA_DIR containing one seq_eval dataset: {roots}')
DATA=roots[0]; table=pd.read_csv(DATA/'samples.csv')
if SETS: table=table[table['set'].isin(SETS)].copy()
if table.empty or table.duplicated(['set','k']).any(): raise ValueError('Empty or duplicate (set,k) rows')
if not np.isfinite(table[['k','map_x','map_y','dist_m']].to_numpy(float)).all(): raise ValueError('Nonfinite coordinates')
for st,g in table.groupby('set'):
    if Path(st).name!=st: raise ValueError('Invalid set name')
    g=g.sort_values('k')
    if len(g)<2 or (np.diff(g.dist_m)<0).any(): raise ValueError(f'Invalid sequence: {st}')
    city=cv2.imread(str(DATA/'maps'/f'{st}.jpg'))
    if city is None or city.shape!=(4096,4096,3): raise ValueError(f'Missing/invalid map: {st}')
    for k in g.k:
        if int(k)!=k: raise ValueError('k must be integer')
        path=DATA/st/f'k{int(k):03d}_uav.jpg'
        im=cv2.imread(str(path))
        if im is None or im.shape!=(256,256,3): raise ValueError(f'Missing/invalid UAV image: {path}')
print('DATA:',DATA,'segments:',table['set'].nunique(),'frames:',len(table))
save_json(RUN/'data_audit.json',dict(data=str(DATA),samples_sha256=hash_file(DATA/'samples.csv'),
    segments=int(table['set'].nunique()),frames=len(table),sets=SETS,seed=SEED,odo=ODO))

if table['set'].nunique()!=92 or len(table)!=1112: raise ValueError('Expected 92 segments and 1112 frames')

In [ ]:
def make_env(name, packages):
    import sys
    import importlib.metadata as metadata
    env=WORK/'envs'/name
    py=env/'bin/python'
    env.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/(name+'_environment.log')

    def command(args, required=True):
        result=subprocess.run([str(x) for x in args],stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True)
        output=result.stdout or ''
        with log.open('a',encoding='utf-8') as f:
            f.write(repr([str(x) for x in args])+chr(10)+output+chr(10))
        if output: print(output,flush=True)
        if required and result.returncode:
            raise RuntimeError(f'Environment command failed ({result.returncode}). See {log}: '+output[-4000:])
        return result

    def valid_python():
        if not py.exists(): return False
        try:
            probe=command([py,'-c',
                'import sys; from pathlib import Path; '
                'assert Path(sys.prefix).resolve()==Path(sys.argv[1]).resolve(); '
                'assert sys.prefix != sys.base_prefix',env],required=False)
            return probe.returncode==0
        except OSError:
            return False

    if not valid_python():
        # Kaggle may omit ensurepip. Create the interpreter without invoking it.
        created=command([sys.executable,'-m','venv','--without-pip',
                         '--system-site-packages',env],required=False)
        if created.returncode or not valid_python():
            # Fallback also handles installations without the stdlib venv module.
            bootstrap=WORK/'virtualenv_bootstrap'
            command([sys.executable,'-m','pip','install','--target',bootstrap,'virtualenv'])
            launcher='import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module("virtualenv",run_name="__main__")'
            command([sys.executable,'-c',launcher,bootstrap,'--system-site-packages',env])
        if not valid_python():
            raise RuntimeError(f'Invalid environment interpreter: {py}. See {log}')
    # Parent pip can install into a pip-less environment; no ensurepip needed.
    pip_check=command([py,'-m','pip','--version'],required=False)
    if pip_check.returncode:
        command([sys.executable,'-m','pip','--python',py,'install','pip'])
    command([py,'-m','pip','--version'])
    pins=[package+'=='+metadata.version(package) for package in ('torch','torchvision','numpy')]
    constraint=env/'constraints.txt'
    constraint.write_text(chr(10).join(pins),encoding='utf-8')
    command([py,'-m','pip','install','-c',constraint,*packages])
    frozen=command([py,'-m','pip','freeze'])
    (RUN/(name+'_pip-freeze.txt')).write_text(frozen.stdout,encoding='utf-8')
    return py


PY=make_env('loma',['vismatch==1.3.1','scipy'])

In [ ]:
sources={'imatch/blocks.py': '"""파이프라인 ⑥: 후보 블록.\n\n좌표 규칙\n- 도시 지도 4096 px. 블록 (bx, by)는 지도의 [by*256, by*256+512) 행, [bx*256, bx*256+512) 열 (x가 열).\n- 블록은 256 px(2단위)씩 어긋나 있어 이웃과 절반씩 겹친다. 위치 범위는 0 <= bx, by < 15.\n- 후보 (bx+dx, by+dy) 기준 좌표 x\' 와 (bx, by) 기준 좌표 x 의 관계: x = x\' + 2 dx, y = y\' + 2 dy.\n"""\nN_BLOCK = 15\nUNITS_PER_BLOCK = 2  # 이웃 블록 사이 간격 (블록 단위)\n\n\ndef crop_block(city, bx, by):\n    """도시 지도에서 블록 (bx, by)를 512 px로 자른다."""\n    return city[by * 256:by * 256 + 512, bx * 256:bx * 256 + 512]\n\n\ndef neighbors(bx, by, radius):\n    """(dx, dy) 목록: 지도 범위 안의 (2 radius + 1)^2 후보."""\n    return [(dx, dy) for dx in range(-radius, radius + 1) for dy in range(-radius, radius + 1)\n            if 0 <= bx + dx < N_BLOCK and 0 <= by + dy < N_BLOCK]\n', 'imatch/geometry.py': '"""파이프라인 ④⑤: 대응점 검증(MAGSAC++)과 호모그래피 → 위치·heading.\n\n좌표 규칙 (Bearing-UAV와 같음)\n- 드론 영상 256 px, 지도 블록 512 px (2x2 타일). 블록 중심이 (0, 0), 1단위 = 128 px = 32 m.\n- 정답 변환: block_px = C + R(theta) (uav_px - 128),  R(t) = [[cos t, sin t], [-sin t, cos t]]\n"""\nimport numpy as np\n\nBLOCK_CENTER = 256.0\nPX_PER_UNIT = 128.0\nM_PER_UNIT = 32.0\n\nRANSAC_PX = 3.0  # MAGSAC++ 임계값(px). scripts/run.py --ransac-px 로 바꾼다\n\n\ndef set_ransac_px(px):\n    global RANSAC_PX\n    RANSAC_PX = float(px)\n\n\ndef rot(a):\n    t = np.deg2rad(a); c, s = np.cos(t), np.sin(t)\n    return np.array([[c, s], [-s, c]])\n\n\ndef turn(img, a):\n    """영상을 a도 돌린다 (반시계, cv2 규칙). 빈 모서리는 검은색."""\n    import cv2\n    h, w = img.shape[:2]\n    return cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1), (w, h), flags=cv2.INTER_LINEAR,\n                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n\n\ndef wrap(d):\n    """각도 차이의 절댓값 (0~180도)."""\n    return np.abs((np.asarray(d) + 180) % 360 - 180)\n\n\ndef magsac(p0, p1):\n    """대응점 → (p0, p1, 호모그래피 H, inlier 마스크). 점이 4개 미만이면 H = None."""\n    import cv2\n    if len(p0) < 4:\n        return p0, p1, None, np.zeros(len(p0), bool)\n    H, mask = cv2.findHomography(p0, p1, cv2.USAC_MAGSAC, RANSAC_PX, maxIters=2000, confidence=0.995)\n    return p0, p1, H, (mask[:, 0].astype(bool) if mask is not None else np.zeros(len(p0), bool))\n\n\ndef pose_from_homography(H, scale, angle, uav_center):\n    """돌린 드론 영상 → 블록 호모그래피에서 위치(x, y: 블록 단위)와 heading(도), 축척을 구한다.\n\n    위치: 드론 영상 중심을 블록으로 옮긴 점. heading: 중심에서의 국소 선형 변환 L의 회전 성분.\n    회전 탐색으로 영상을 angle만큼 돌렸으므로 L에 R(angle)을 곱해 원래 영상 기준으로 되돌린다.\n    """\n    S = np.diag([scale, scale, 1.])\n    Hb = np.linalg.inv(S) @ H @ S                                   # 축척 보정 (base px 기준)\n    h = Hb @ np.array([uav_center[0], uav_center[1], 1.]); c = h[:2] / h[2]  # 회전해도 중심은 그대로\n    L = (Hb[:2, :2] - np.outer(c, Hb[2, :2])) / h[2] @ rot(angle)   # 중심에서의 야코비안\n    return dict(px=float((c[0] - BLOCK_CENTER) / PX_PER_UNIT), py=float((c[1] - BLOCK_CENTER) / PX_PER_UNIT),\n                head=float(np.degrees(np.arctan2(L[0, 1], L[0, 0]))), scale_est=float(np.sqrt(abs(np.linalg.det(L)))))\n', 'imatch/matchers.py': '"""파이프라인 ②③: 특징 추출과 매칭. 모든 매처는 (드론 영상, 블록) → magsac() 결과를 돌려준다.\n\n새 매처를 실험하려면 __call__(im0, im1) 을 가진 클래스를 만들고 get_matcher()에 이름을 추가하면 된다.\n"""\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import magsac\n\n# vismatch의 get_matcher에 없는 LoMa 변형은 직접 만든다 (\'loma\' = LoMa-B는 get_matcher로)\nLOMA_ARCH = {\'loma-b128\': \'LoMa-B128\', \'loma-l\': \'LoMa-L\', \'loma-g\': \'LoMa-G\'}\n\nclass VisMatch:\n    """vismatch 라이브러리의 공개 매처 (loma, romav2, sift-lightglue 등). GPU 권장.\n\n    LoMa 변형: \'loma\'(= LoMa-B, 주력), \'loma-b128\'(경량), \'loma-l\', \'loma-g\', \'loma-r\'.\n    """\n\n    def __init__(self, name, device=\'cuda\'):\n        import torch\n        self.name, self.torch = name, torch\n        if name in LOMA_ARCH:\n            from vismatch.im_models import loma\n            self.m = loma.LoMaMatcher(device, 2048, arch=LOMA_ARCH[name])\n        else:\n            from vismatch import get_matcher\n            self.m = get_matcher(name, device=device)\n        if name.startswith(\'loma\') and os.environ.get(\'LOMA_FP32\'):\n            # GTX 1650은 fp16에서 키포인트가 전부 NaN이 되어 오류 없이 짝 0개가 나온다. LOMA_FP32=1로 fp32를 쓴다.\n            # T4 결과(fp16)와 조금 다를 수 있으니 보고용 숫자는 T4 기본 설정으로 낸다\n            from vismatch.utils import force_float32\n            force_float32(self.m.matcher)\n        if name == \'romav2\':\n            # 기본 \'precise\' 설정은 T4(bf16 없음)에서 메모리 부족으로 멈췄다. 우리 영상은 256 / 512 px라\n            # \'base\' 설정(640 px, 단방향, fp32)을 쓴다.\n            from vismatch.im_models import romav2 as rv2\n            rv2._disable_autocast()\n            self.m.romav2_model.apply_setting(os.environ.get(\'ROMAV2_SETTING\', \'base\'))\n            self.name = f"romav2-{os.environ.get(\'ROMAV2_SETTING\', \'base\')}"\n\n    def __call__(self, im0, im1):\n        t = lambda im: self.torch.from_numpy(im).permute(2, 0, 1).float() / 255.\n        r = self.m(t(im0), t(im1))\n        p0, p1 = r[\'matched_kpts0\'], r[\'matched_kpts1\']\n        # vismatch는 inlier 점만 주고 마스크는 안 주므로 같은 점으로 MAGSAC++를 다시 돌린다\n        return magsac(np.float32(p0).reshape(-1, 2), np.float32(p1).reshape(-1, 2))\n\n\nclass CvSift:\n    """GPU 없이 도는 기준선: OpenCV SIFT + Lowe 비율 검사 + MAGSAC++. 설치·동작 확인용."""\n    name = \'cv-sift\'\n\n    def __init__(self):\n        self.sift = cv2.SIFT_create(4096); self.bf = cv2.BFMatcher()\n\n    def __call__(self, im0, im1):\n        k0, d0 = self.sift.detectAndCompute(cv2.cvtColor(im0, cv2.COLOR_RGB2GRAY), None)\n        k1, d1 = self.sift.detectAndCompute(cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY), None)\n        if d0 is None or d1 is None or len(k0) < 2 or len(k1) < 2:\n            return np.empty((0, 2)), np.empty((0, 2)), None, np.zeros(0, bool)\n        mm = [m for m, n in self.bf.knnMatch(d0, d1, k=2) if m.distance < 0.8 * n.distance]\n        p0 = np.float32([k0[m.queryIdx].pt for m in mm]).reshape(-1, 2)\n        p1 = np.float32([k1[m.trainIdx].pt for m in mm]).reshape(-1, 2)\n        return magsac(p0, p1)\n\n\nclass VggMNN:\n    """비교 실험 \'VGG16 백본은 그대로 두면?\' (docs/02_실험기록.md 7장). 원 Bearing-UAV 체크포인트의 고정\n    VGG16 conv4_3(B2가 상관을 계산하는 그 특징)을 점 단위 기술자로 쓴다.\n\n    특징 맵의 칸(stride 8, 중심 8 i + 4)이 점이다. 블록 평균을 뺀 코사인 유사도로 상호 최근접 짝을 짓고,\n    이후는 다른 매처와 같은 MAGSAC++ / 호모그래피를 쓴다. 회전으로 생긴 검은 모서리의 칸은 뺀다.\n    capstone1 저장소와 체크포인트가 필요하다:\n      CAPSTONE1_DIR=<capstone1 경로> VGG_CKPT=Bearing_UAV/cross_view python scripts/run.py --matchers vgg-mnn --device cpu\n    결과: Singapore 44%(B2 69%, 100샘플) → 원 모델 특징으로는 점 단위 매칭이 안 된다.\n    """\n    name = \'vgg-mnn\'\n\n    def __init__(self):\n        import sys\n        import torch\n        root = os.environ[\'CAPSTONE1_DIR\']\n        sys.path.insert(0, root)\n        from diag.diag_modules import DEV, load_model, to_tensor\n        self.torch, self.tt, self.dev = torch, to_tensor, DEV\n        self.bb = load_model(os.path.join(root, os.environ.get(\'VGG_CKPT\', \'Bearing_UAV/cross_view\'))).backbone\n\n    def _feat(self, im):\n        with self.torch.no_grad():\n            f = self.bb(self.tt(im).unsqueeze(0).to(self.dev))[0]          # [C, h, w]\n        C, h, w = f.shape; st = im.shape[0] / h\n        yy, xx = np.mgrid[0:h, 0:w]\n        pts = np.stack([xx.ravel() * st + st / 2, yy.ravel() * st + st / 2], 1).astype(np.float32)\n        return f.reshape(C, -1).T, pts\n\n    def __call__(self, im0, im1):\n        F = self.torch.nn.functional\n        d0, p0 = self._feat(im0); d1, p1 = self._feat(im1)\n        valid = im0[np.clip(p0[:, 1].astype(int), 0, im0.shape[0] - 1), np.clip(p0[:, 0].astype(int), 0, im0.shape[1] - 1)].sum(1) > 0\n        d0, p0 = d0[self.torch.from_numpy(valid)], p0[valid]\n        mu = d1.mean(0, keepdim=True)\n        d0, d1 = F.normalize(d0 - mu, dim=1), F.normalize(d1 - mu, dim=1)\n        S = d0 @ d1.T\n        j = S.argmax(1); i = S.argmax(0)\n        mutual = (i[j] == self.torch.arange(len(j), device=j.device)).cpu().numpy()\n        j = j.cpu().numpy()\n        return magsac(p0[mutual], p1[j[mutual]])\n\n\ndef get_matcher(name, device=\'cuda\'):\n    if name == \'cv-sift\':\n        return CvSift()\n    if name == \'vgg-mnn\':\n        return VggMNN()\n    return VisMatch(name, device)\n', 'imatch/metrics.py': '"""팀 공통 평가 규칙 (2026-10-15 확정). 모든 표의 숫자는 이 규칙으로 계산한다.\n\n- 추정은 2x2 블록 안으로 clip 한다. 추정이 없으면 블록 중심 (0, 0)을 답으로 낸다.\n  (B2·CorrMatchNet은 원래 블록 안만 답할 수 있으므로 같은 조건을 맞춘 것)\n- 실패 = 추정 없음 또는 블록 밖으로 0.1단위(3.2 m) 넘게 나감.\n- 성공 = 위치 오차 10 m 이내.\n- 비교는 같은 샘플끼리 짝지어 McNemar 정확 검정 (나만 성공 / 상대만 성공).\n"""\nimport numpy as np\nfrom scipy.stats import binomtest\n\nfrom .blocks import UNITS_PER_BLOCK\nfrom .geometry import M_PER_UNIT\n\n\ndef errors(pred_x, pred_y, gt_x, gt_y, lim=1.0):\n    """(위치 오차 m, 실패 여부) 배열. lim = clip 범위 (블록 단위)."""\n    px, py = np.asarray(pred_x, float), np.asarray(pred_y, float)\n    found = np.isfinite(px) & np.isfinite(py)\n    px, py = np.nan_to_num(px), np.nan_to_num(py)\n    fail = ~found | (np.abs(px) > lim + 0.1) | (np.abs(py) > lim + 0.1)\n    e = np.hypot(np.clip(px, -lim, lim) - np.asarray(gt_x), np.clip(py, -lim, lim) - np.asarray(gt_y)) * M_PER_UNIT\n    return e, fail\n\n\ndef errors_chosen_block(pred_x, pred_y, dx, dy, gt_x, gt_y):\n    """후보 블록 결과용 errors. pred는 고른 블록(정답 블록 + (dx, dy)) 기준이고, clip은 그 블록의 영상 범위(±2)로 한다.\n\n    errors의 기본 ±1은 정답 블록일 때만 맞다. 정답 블록 좌표로 옮겨서 ±1로 clip하면 이웃 블록을 고른 추정이 정답\n    쪽으로 끌려와 오차가 작게 나오고, 고른 블록 기준 ±1로 clip하면 이웃 블록의 맞는 추정(예: -1.45)이 틀어진다.\n    이 함수의 숫자는 errors로 잰 표와 바로 비교하면 안 된다. 같은 후보 결과끼리만 비교한다.\n    """\n    dx, dy = np.nan_to_num(np.asarray(dx, float)), np.nan_to_num(np.asarray(dy, float))\n    return errors(pred_x, pred_y, np.asarray(gt_x) - UNITS_PER_BLOCK * dx, np.asarray(gt_y) - UNITS_PER_BLOCK * dy, lim=2.0)\n\n\ndef mcnemar(ok_mine, ok_other):\n    """\'나만 성공 / 상대만 성공, p\' 문자열."""\n    a, b = int((ok_mine & ~ok_other).sum()), int((~ok_mine & ok_other).sum())\n    return f"{a} / {b}, p = {binomtest(a, a + b).pvalue:.2g}" if a + b else "0 / 0"\n', 'imatch/nav.py': '"""닫힌 루프 1 km 항법. capstone1 diag/diag_phase0.py의 make_traj, Cropper, nav_run을 옮긴 것이다.\n\n바뀐 점은 드론 영상을 frames(x, y, heading) → (영상, 영상의 실제 x, y)로 받는 것뿐이다.\n- Crop2D: 기존 T5와 같다. 지도를 잘라 드론 영상으로 쓴다 (2D는 SIFT로도 풀리는 시험이라 동작 확인용)\n- Nearest3D: 궤적 지점에서 가장 가까운 3D 렌더를 쓴다. 연속 프레임이 없어서 만든 근사다.\n  렌더는 도시당 22,500장이고 궤적까지 거리 중앙값 3.0 m, 최대 약 11 m. heading은 렌더마다 무작위라\n  비행 방향과 맞지 않는다. 오차는 렌더의 실제 위치로 잰다.\n"""\nimport hashlib\nimport os\nimport re\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial import cKDTree\n\nfrom .blocks import N_BLOCK, crop_block, neighbors\nfrom .geometry import PX_PER_UNIT\nfrom .pipeline import localize\n\nPX = 4.0  # px per metre (0.25 m/px)\nW = 384\nMETA_3D = \'Bearing_UAV_90K/c4m_254k_96bc_b15_s100_v3d/metadata/metadata.csv\'\n\n\ndef make_traj(n_traj, dist_m, seed, n_block=15):\n    rng = np.random.default_rng(seed)\n    lo, hi = 300, 256 * (n_block + 1) - 300\n    starts, goals = [], []\n    while len(starts) < n_traj:\n        s = rng.uniform(lo, hi, 2); a = rng.uniform(0, 2 * np.pi)\n        g = s + dist_m * PX * np.array([np.cos(a), np.sin(a)])\n        if (g > lo).all() and (g < hi).all():\n            starts.append(s); goals.append(g)\n    return np.array(starts), np.array(goals)\n\n\nclass Crop2D:\n    """지도에서 드론 영상을 자른다 (capstone1 Cropper와 같은 결과)."""\n    def __init__(self, city):\n        self.pad = cv2.copyMakeBorder(city, W, W, W, W, cv2.BORDER_REFLECT)\n\n    def __call__(self, cx, cy, theta):\n        x0, y0 = int(cx) - W, int(cy) - W\n        win = self.pad[y0 + W:y0 + 3 * W, x0 + W:x0 + 3 * W]\n        lx, ly = cx - x0, cy - y0\n        R = cv2.getRotationMatrix2D((lx, ly), -theta, 1)\n        rot = cv2.warpAffine(win, R, (win.shape[1], win.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)\n        c_x, c_y = int(lx), int(ly)\n        return rot[c_y - 128:c_y + 128, c_x - 128:c_x + 128], cx, cy\n\n\nclass Nearest3D:\n    """가장 가까운 3D 렌더를 돌려준다. capstone1의 Bearing_UAV_90K가 필요하다."""\n    def __init__(self, capstone1, st):\n        ho = re.search(r\'(\\d+bc)\', st).group(1)\n        df = pd.read_csv(os.path.join(capstone1, META_3D), usecols=[\'block_x\', \'block_y\', \'x_norm\', \'y_norm\', \'target_path\'])\n        df = df[df.target_path.str.contains(f\'_{ho}_\')]\n        self.xy = np.c_[df.block_x * 256 + 256 + df.x_norm * PX_PER_UNIT, df.block_y * 256 + 256 + df.y_norm * PX_PER_UNIT]\n        self.paths = [os.path.join(capstone1, p) for p in df.target_path]\n        self.tree = cKDTree(self.xy)\n\n    def __call__(self, cx, cy, theta):\n        i = self.tree.query((cx, cy))[1]\n        return cv2.cvtColor(cv2.imread(self.paths[i]), cv2.COLOR_BGR2RGB), self.xy[i, 0], self.xy[i, 1]\n\n\nclass MatchLoc:\n    """localize를 nav_run의 loc(crops, headings, blocks) 형식으로 감싼다. 위치는 블록 단위 (1 = 128 px).\n\n    추정이 없으면 블록 중심을 답한다 (imatch/metrics.py 규칙과 같음). 나침반은 쓰지 않으므로 headings는 무시한다.\n    걸음별 기록은 self.last에 남고, nav_run의 log에 들어간다.\n    같은 (영상, 블록)은 다시 매칭하지 않는다 (self.cache). R0~R4를 한 프로세스에서 돌리면 겹치는 걸음을 아낀다.\n    stop_inl > 0이면 각도 탐색을 inlier가 stop_inl 이상인 첫 각도에서 멈춘다 (localize 조기 종료).\n    """\n    def __init__(self, matcher, city, scale=1.0, angles=(0.0,), stop_inl=0):\n        self.matcher, self.city, self.scale, self.angles, self.last, self.cache = matcher, city, scale, angles, [], {}\n        self.stop_inl = stop_inl\n\n    def __call__(self, crops, headings, blocks):\n        out, self.last = [], []\n        for crop, (bx, by) in zip(crops, blocks):\n            xy, rec = self.one(crop, bx, by)\n            out.append(xy); self.last.append(rec)\n        return np.array(out)\n\n    def one(self, crop, bx, by):\n        """블록 하나: (위치(블록 단위), 기록). self.last는 건드리지 않는다 (Research가 nav_run 걸음 중간에 부른다)."""\n        key = (hashlib.md5(crop.tobytes()).hexdigest(), bx, by)\n        if key not in self.cache:\n            recs, best = localize(self.matcher, crop, crop_block(self.city, bx, by), (0.0, 0.0, 0.0), self.scale, self.angles,\n                                  stop_inl=self.stop_inl)\n            self.cache[key] = ((best[\'px\'], best[\'py\']) if best else (0.0, 0.0),\n                               dict(n_inl=best[\'n_inl\'] if best else 0, found=int(best is not None), sec=sum(r[\'sec\'] for r in recs),\n                                    n_ang=len(recs)))\n        return self.cache[key]\n\n\nclass PreviousAngleLoc(MatchLoc):\n    """R1 replay: try the preceding accepted rotation first for one trajectory."""\n    previous_angle = None\n\n    def one(self, crop, bx, by):\n        order = self.angles\n        if self.previous_angle is not None:\n            order = [self.previous_angle] + [a for a in order if a != self.previous_angle]\n        records, best = localize(self.matcher, crop, crop_block(self.city, bx, by),\n                                 (0., 0., 0.), self.scale, order, stop_inl=self.stop_inl)\n        self.previous_angle = best[\'angle\'] if best and best[\'n_inl\'] >= 15 else None\n        xy = (best[\'px\'], best[\'py\']) if best else (0., 0.)\n        return xy, dict(n_inl=best[\'n_inl\'] if best else 0, found=int(best is not None),\n                        sec=sum(r[\'sec\'] for r in records), n_ang=len(records))\n\n\nclass NoMatch:\n    """매칭하지 않는 기준선. 항상 블록 중심을 답하고 inlier 0을 돌려준다.\n\n    reject_inl = 0이면 "항상 중앙", reject_inl > 0이면 모든 걸음이 "모름"이라 추측 항법만으로 난다.\n    """\n    def __call__(self, crops, headings, blocks):\n        self.last = [dict(n_inl=0, found=0, sec=0.0) for _ in crops]\n        return np.zeros((len(crops), 2))\n\n\nclass Research:\n    """R3·R4: "모름"이면 추측 위치 주변 블록을 다시 매칭한다. nav_run의 recover로 넘긴다.\n\n    R3: 추측 위치의 블록과 십자 이웃 4개 (⑥ 블록 선택의 후보 5개, experiments/2026-11-06_block_select).\n    R4 (wide_after > 0): 연속 "모름"이 wide_after걸음 이상이면 3×3 9개.\n    통과 = 그 블록 기준 추정이 ±inside 안 (inside 규칙)이고 inlier >= min_inl. 여러 개면 추측 위치에 가장 가까운 것, 없으면 None.\n    gate_m > 0이면 십자 재탐색의 답도 추측 위치에서 gate_m m 안이어야 한다 (R2와 같은 움직임 검사).\n    넓게 찾기는 추측 위치가 이미 밀렸다고 보고 움직임 검사를 하지 않는다. 대신 wide_inl > 0이면 넓게 찾기에서만\n    inlier >= wide_inl을 요구한다 (R4 학습 도시에서 틀린 3×3 답은 inlier 중앙값 16, 맞은 답은 26).\n    추측 위치의 블록은 이번 걸음에 이미 매칭했으므로 캐시에서 나온다. 부를 때마다 self.log에 한 줄 남긴다\n    (rec_sec = 이번 걸음에 더 든 매칭 시간, 추측 위치의 블록 제외).\n    """\n    def __init__(self, match, min_inl=15, gate_m=0., wide_after=0, inside=1.1, wide_inl=0):\n        self.match, self.min_inl, self.gate_m, self.wide_after, self.inside = match, min_inl, gate_m, wide_after, inside\n        self.wide_inl = wide_inl\n        self.log = []\n\n    def __call__(self, i, crop, guess, streak):\n        wide = self.wide_after > 0 and streak >= self.wide_after\n        min_inl = max(self.min_inl, self.wide_inl) if wide else self.min_inl\n        bx = int(np.clip(np.round((guess[0] - 256) / 256), 0, N_BLOCK - 1))\n        by = int(np.clip(np.round((guess[1] - 256) / 256), 0, N_BLOCK - 1))\n        cands = [(dx, dy) for dx, dy in neighbors(bx, by, 1) if wide or abs(dx) + abs(dy) <= 1]\n        best, best_d, n_pass, sec = None, np.inf, 0, 0.\n        for dx, dy in cands:\n            (x, y), rec = self.match.one(crop, bx + dx, by + dy)\n            sec += rec[\'sec\'] if (dx, dy) != (0, 0) else 0.\n            if not rec[\'found\'] or rec[\'n_inl\'] < min_inl or max(abs(x), abs(y)) > self.inside:\n                continue\n            p = np.array([(bx + dx) * 256 + 256, (by + dy) * 256 + 256]) + np.array([x, y]) * PX_PER_UNIT\n            d = np.linalg.norm(p - guess) / PX\n            if self.gate_m > 0 and not wide and d > self.gate_m:\n                continue\n            n_pass += 1\n            if d < best_d:\n                best, best_d, pick = p, d, (bx + dx, by + dy, rec[\'n_inl\'])\n        self.log.append(dict(traj=int(i), rec_wide=int(wide), rec_cand=len(cands), rec_pass=n_pass, rec_sec=sec,\n                             **(dict(zip((\'rec_bx\', \'rec_by\', \'rec_n_inl\'), pick)) if best is not None else {})))\n        return best\n\n\nODO = {\'none\': None, \'vio\': dict(bias_deg=3.0, head_deg=2.0, scale=0.05)}\n"""이동 오차 (추측 항법이 공짜로 맞지 않게).\n\n시뮬레이션의 드론은 명령한 대로 정확히 움직이므로, 오차가 없으면 영상을 안 쓰고 추측 항법만 해도 100% 도착한다.\n\'vio\': 궤적마다 heading 치우침 N(0, 3°), 걸음마다 heading N(0, 2°), 걸음 길이 N(1, 5%).\n1 km에서 heading 3° 치우침만으로 옆으로 약 52 m 밀린다. 드론은 이 오차를 모르고 명령한 이동량으로 추측한다.\n"""\n\n\ndef nav_run(loc, frames, starts, goals, decode_px=128, oracle_block=False, step_m=15., arrive_m=20., n_block=15, log=None,\n            odo=None, seed=0, reject_inl=0, gate_m=0., recover=None):\n    """(성공 여부, 종료 이유, 궤적별 걸음 오차 중앙값 m). log: 걸음마다 기록을 덧붙일 목록.\n\n    복구 (docs/04 R0~R4). 기본값이면 R0이고 예전 결과와 같다.\n    odo: ODO의 값. 실제 이동에 오차를 넣는다. 같은 seed면 모든 R에서 같은 오차가 나온다.\n    reject_inl: 고른 답의 inlier가 이보다 작으면 "모름" (R1, 15).\n    gate_m: 추정이 추측 위치(직전 위치 + 명령 이동량)에서 이보다 멀면 "모름" (R2).\n    recover: "모름"일 때 부르는 함수 recover(i, crop, guess_px, streak) → 지도 px 위치 또는 None (R3, R4 자리).\n             None을 돌려주거나 recover가 없으면 추측 위치를 쓴다. streak = 이 궤적의 연속 "모름" 걸음 수 (이번 걸음 포함).\n    """\n    n = len(starts); dist_m = np.linalg.norm(goals[0] - starts[0]) / PX\n    r = starts.copy(); nom = r.copy()\n    alive = np.ones(n, bool); success = np.zeros(n, bool); reason = np.array([\'timeout\'] * n, dtype=object)\n    heading = np.full(n, 90.0); err = [[] for _ in range(n)]; streak = np.zeros(n, int)\n    T = int(3 * dist_m / step_m)\n    rng = np.random.default_rng(seed + 1)\n    bias = rng.normal(0, odo[\'bias_deg\'], n) if odo else np.zeros(n)\n    dh = rng.normal(0, odo[\'head_deg\'], (n, T)) if odo else np.zeros((n, T))\n    ds = rng.normal(0, odo[\'scale\'], (n, T)) if odo else np.zeros((n, T))\n    for t in range(T):\n        idx = np.where(alive)[0]\n        if len(idx) == 0:\n            break\n        blocks, crops, cen, real = [], [], [], []\n        for i in idx:\n            ref = r[i] if oracle_block else nom[i]\n            bx = int(np.clip(np.round((ref[0] - 256) / 256), 0, n_block - 1))\n            by = int(np.clip(np.round((ref[1] - 256) / 256), 0, n_block - 1))\n            blocks.append((bx, by)); cen.append(np.array([bx * 256 + 256, by * 256 + 256]))\n            crop, fx, fy = frames(r[i, 0], r[i, 1], heading[i])\n            crops.append(crop); real.append(np.array([fx, fy]))\n        pos = loc(crops, heading[idx], blocks)\n        last = getattr(loc, \'last\', None) or [{} for _ in idx]\n        for j, i in enumerate(idx):\n            raw = cen[j] + pos[j] * decode_px\n            guess = nom[i].copy()  # 드론이 아는 것만으로 추측한 이번 위치 (복사: 아래에서 nom[i]를 고친다)\n            n_inl = last[j].get(\'n_inl\', np.inf)\n            unknown = (reject_inl > 0 and n_inl < reject_inl) or (gate_m > 0 and np.linalg.norm(raw - guess) / PX > gate_m)\n            streak[i] = streak[i] + 1 if unknown else 0\n            p, how = raw, \'match\'\n            if unknown:\n                q = recover(i, crops[j], guess, streak[i]) if recover else None\n                p, how = (np.asarray(q, float), \'recover\') if q is not None else (guess, \'guess\')\n                if q is not None: streak[i] = 0\n            err[i].append(float(np.linalg.norm(p - real[j]) / PX))\n            if log is not None:\n                log.append(dict(traj=int(i), step=t, real_x=float(real[j][0]), real_y=float(real[j][1]), est_x=float(p[0]),\n                                est_y=float(p[1]), block_x=blocks[j][0], block_y=blocks[j][1], err_m=err[i][-1],\n                                outside_block=int(np.abs(real[j] - cen[j]).max() > 128),\n                                raw_err_m=float(np.linalg.norm(raw - real[j]) / PX), unknown=int(unknown), used=how,\n                                **last[j]))\n            d = goals[i] - p; u = d / max(np.linalg.norm(d), 1e-6)\n            heading[i] = np.degrees(np.arctan2(-u[1], u[0]))\n            a = np.radians(bias[i] + dh[i, t]); c, s_ = np.cos(a), np.sin(a)\n            move = np.array([c * u[0] - s_ * u[1], s_ * u[0] + c * u[1]]) * (1 + ds[i, t])\n            r[i] = r[i] + move * step_m * PX; nom[i] = p + u * step_m * PX\n            real_d = np.linalg.norm(r[i] - goals[i]) / PX; pred_d = np.linalg.norm(p - goals[i]) / PX\n            if real_d < arrive_m or pred_d < arrive_m:\n                alive[i] = False; success[i] = real_d < 2 * arrive_m\n                reason[i] = \'arrived_ok\' if success[i] else \'arrived_pred_only\'\n            elif (r[i] < 200).any() or (r[i] > 256 * (n_block + 1) - 200).any():\n                alive[i] = False; reason[i] = \'out_of_map\'\n    return success, reason, [float(np.median(e)) for e in err if e]\n', 'imatch/pipeline.py': '"""파이프라인 ①~⑤를 한 샘플에 적용한다.\n\n① 회전 탐색: 드론 영상을 angles의 각 각도로 돌린다 (angles=[0]이면 탐색 없음, 원 Bearing-UAV와 같은 조건)\n②③ 매칭: matcher(돌린 드론 영상, 블록)\n④ 검증: MAGSAC++ inlier\n⑤ 위치·heading: 호모그래피 → pose_from_homography\n→ inlier가 가장 많은 각도의 결과를 고른다.\n   stop_inl > 0이면 각도를 차례로 해 보다가 inlier가 stop_inl 이상인 각도가 나오면 거기서 멈춘다 (조기 종료, 빠르게).\n"""\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import pose_from_homography, rot, turn\n\n\ndef localize(matcher, uav, block, gt, scale=1.0, angles=(0.0,), px_tol=8.0, stop_inl=0):\n    """각도별 기록 목록과 고른 결과(best, 없으면 None)를 돌려준다.\n\n    gt = (x_norm, y_norm, theta): 위치 추정에는 쓰지 않고, 대응점이 맞는지(good_*) 채점하는 데만 쓴다.\n    """\n    hu, wu = uav.shape[:2]; cu = np.array([wu / 2, hu / 2])\n    U = cv2.resize(uav, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else uav\n    B = cv2.resize(block, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else block\n    xn, yn, th = gt; C = np.array([256 + xn * 128, 256 + yn * 128]); Rg = rot(th)\n    recs = []\n    for a in angles:\n        t0 = time.time()\n        p0, p1, H, inl = matcher(turn(U, a), B)\n        dt = time.time() - t0\n        p0 = np.asarray(p0, np.float64) / scale; p1 = np.asarray(p1, np.float64) / scale\n        # 원래 드론 영상 좌표로 되돌림: p_turned = cu + R(a) (p - cu)  ->  p = cu + R(a)^T (p_turned - cu)\n        po = (p0 - cu) @ rot(a) + cu if len(p0) else p0\n        good = np.linalg.norm(C + (po - cu) @ Rg.T - p1, axis=1) < px_tol if len(p0) else np.zeros(0, bool)\n        r = dict(angle=a, n_match=int(len(p0)), n_inl=int(inl.sum()), sec=dt,\n                 good_match=float(good.mean()) if len(good) else np.nan,\n                 good_inl=float(good[inl].mean()) if inl.sum() else np.nan)\n        if H is not None and inl.sum() >= 4:\n            r.update(pose_from_homography(H, scale, a, cu))\n        recs.append(r)\n        if stop_inl > 0 and \'px\' in r and r[\'n_inl\'] >= stop_inl:\n            break\n    ok = [r for r in recs if \'px\' in r]\n    best = max(ok, key=lambda r: r[\'n_inl\']) if ok else None\n    return recs, best\n', 'imatch/replay.py': '"""실제 비행 재생 항법 (UAV-VisLoc seq_eval). nav.py의 닫힌 루프 시뮬레이터와 따로 둔다.\n\n실제 사진은 드론이 날아간 경로에서만 찍혀 있어, 추정대로 드론을 움직이고 그 자리 사진을 새로 가져올 수 없다\n(nav.py nav_run의 frames). 그래서 찍힌 순서대로 사진을 주고, 매 걸음\n  추측 위치 = 직전 추정 + 이동량(실제 이동 + VIO 오차),  블록 = 추측 위치를 담은 블록,  추정 = loc(사진, 블록)\n으로 위치를 잇는다 (열린 루프). 정답 GPS는 사진 순서와 채점, 이동량 시뮬레이션에만 쓴다. 출발 위치는 정답으로 준다.\n\n모든 방법이 같은 구간·출발점·이동 오차(같은 seed)·블록 고르기 규칙을 쓰므로 짝지어 비교할 수 있다.\nloc은 nav_run과 같은 형식이다: loc(crops, headings, blocks) → (n, 2) 블록 단위 위치, 선택으로 loc.last = [{n_inl, ...}].\n그래서 기준선(B1, CorrMatchNet 등)은 3D 항법에 꽂았던 loc을 지도만 바꿔 그대로 쓴다:\n\n  from imatch.replay import run_segments, summarize\n  steps, segs = run_segments(lambda city: B1Loc(model, city), \'data/visloc/seq_eval\', odo=\'vio\', seed=2026)\n  # B1Loc: 3D 항법 때 b1_loc_sg를 만든 클래스. 구간마다 지도(city)만 바뀐다\n  print(summarize(steps, segs))\n\nnav_run과 다른 점\n- 드론은 정답 경로를 그대로 지나간다. 그래서 "도착 성공" 대신 걸음별 오차와 마지막 오차로 잰다.\n- 이동량 오차는 nav.ODO[\'vio\']와 같은 분포를 실제 이동(사진 사이 GPS 변위)에 넣는다:\n  구간마다 heading 치우침 N(0, 3°), 걸음마다 heading N(0, 2°), 걸음 길이 N(1, 5%).\n- 걸음 간격이 60~130 m다 (3D는 15 m).\n"""\nimport os\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nfrom .blocks import N_BLOCK\nfrom .geometry import PX_PER_UNIT\nfrom .nav import ODO, PX\n\n\ndef _rot(v, deg):\n    a = np.radians(deg); c, s = np.cos(a), np.sin(a)\n    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])\n\n\ndef load_segments(data, sets=None, max_dist=None):\n    """seq_eval/samples.csv → {set: 걸음 표(k 순서)}. max_dist(m)면 구간 시작부터 그 거리까지의 사진만."""\n    S = pd.read_csv(os.path.join(data, \'samples.csv\'))\n    if sets:\n        S = S[S.set.isin(sets)]\n    if max_dist is not None:\n        S = S[S.dist_m <= max_dist + 1e-6]\n    return {st: g.sort_values(\'k\').reset_index(drop=True) for st, g in S.groupby(\'set\', sort=True)}\n\n\ndef replay_one(loc, g, data, st, i, odo=None, seed=0, reject_inl=0, gate_m=0., recover=None, oracle_block=False,\n               n_block=N_BLOCK, log=None):\n    """구간 하나. i = 구간 번호 (이동 오차 난수를 구간마다 고정하는 데 씀). 걸음마다 log에 dict를 덧붙인다."""\n    real = g[[\'map_x\', \'map_y\']].to_numpy(float)\n    rng = np.random.default_rng([seed + 1, i])\n    n = len(g)\n    bias = rng.normal(0, odo[\'bias_deg\']) if odo else 0.0\n    dh = rng.normal(0, odo[\'head_deg\'], n) if odo else np.zeros(n)\n    ds = rng.normal(0, odo[\'scale\'], n) if odo else np.zeros(n)\n    est, streak = real[0].copy(), 0\n    for k in range(n):\n        if k == 0:\n            guess, course = real[0].copy(), 0.0\n        else:\n            d = _rot(real[k] - real[k - 1], bias + dh[k]) * (1 + ds[k])   # 드론이 아는 이동량 (지도 px, y 아래)\n            guess = est + d\n            course = float(np.degrees(np.arctan2(-d[1], d[0])))           # nav_run heading과 같은 규칙\n        ref = real[k] if oracle_block else guess\n        bx = int(np.clip(np.round((ref[0] - 256) / 256), 0, n_block - 1))\n        by = int(np.clip(np.round((ref[1] - 256) / 256), 0, n_block - 1))\n        cen = np.array([bx * 256 + 256, by * 256 + 256], float)\n        crop = cv2.cvtColor(cv2.imread(os.path.join(data, st, f\'k{int(g.k[k]):03d}_uav.jpg\')), cv2.COLOR_BGR2RGB)\n        pos = np.asarray(loc([crop], np.array([course]), [(bx, by)]), float)[0]\n        last = (getattr(loc, \'last\', None) or [{}])[0]\n        raw = cen + pos * PX_PER_UNIT\n        n_inl = last.get(\'n_inl\', np.inf)\n        unknown = (reject_inl > 0 and n_inl < reject_inl) or (gate_m > 0 and np.linalg.norm(raw - guess) / PX > gate_m)\n        streak = streak + 1 if unknown else 0\n        p, how = raw, \'match\'\n        if unknown:\n            q = recover(i, crop, guess, streak) if recover else None\n            p, how = (np.asarray(q, float), \'recover\') if q is not None else (guess, \'guess\')\n            if q is not None:\n                streak = 0\n        est = p\n        if log is not None:\n            log.append(dict(set=st, traj=i, k=int(g.k[k]), dist_m=float(g.dist_m[k]), real_x=real[k, 0], real_y=real[k, 1],\n                            guess_x=guess[0], guess_y=guess[1], est_x=p[0], est_y=p[1], block_x=bx, block_y=by,\n                            err_m=float(np.linalg.norm(p - real[k]) / PX), raw_err_m=float(np.linalg.norm(raw - real[k]) / PX),\n                            guess_err_m=float(np.linalg.norm(guess - real[k]) / PX),\n                            outside_block=int(np.abs(real[k] - cen).max() > 128), unknown=int(unknown), used=how,\n                            tilted=bool(g.tilted[k]) if \'tilted\' in g else False,\n                            turning=bool(g.turning[k]) if \'turning\' in g else False, **last))\n    return est\n\n\ndef run_segments(make_loc, data, sets=None, odo=\'vio\', seed=2026, reject_inl=0, gate_m=0., make_recover=None,\n                 oracle_block=False, max_dist=None, verbose=True):\n    """모든 구간을 재생한다. make_loc(city) → loc, make_recover(loc) → recover 또는 None.\n\n    반환: (걸음 표, 구간 표). 구간 표: 걸음 수, 길이, 마지막 오차, 걸음별 10/20/32 m 비율, 모름 비율.\n    """\n    odo = ODO[odo] if isinstance(odo, str) else odo\n    segs = load_segments(data, sets, max_dist)\n    steps, rows = [], []\n    for i, (st, g) in enumerate(segs.items()):\n        city = cv2.cvtColor(cv2.imread(os.path.join(data, \'maps\', f\'{st}.jpg\')), cv2.COLOR_BGR2RGB)\n        loc = make_loc(city)\n        rec = make_recover(loc) if make_recover else None\n        log = []\n        replay_one(loc, g, data, st, i, odo=odo, seed=seed, reject_inl=reject_inl, gate_m=gate_m, recover=rec,\n                   oracle_block=oracle_block, log=log)\n        if rec is not None and getattr(rec, \'log\', None):\n            L = pd.DataFrame(log)\n            R = pd.DataFrame(rec.log, index=L.index[L.unknown == 1])\n            L = L.join(R.drop(columns=\'traj\'))\n            if \'sec\' in L:\n                L[\'sec\'] += L.rec_sec.fillna(0)\n            log = L.to_dict(\'records\')\n        steps += log\n        e = np.array([s[\'err_m\'] for s in log])\n        rows.append(dict(set=st, traj=i, n_steps=len(log), length_m=float(g.dist_m.iloc[-1]), final_err_m=float(e[-1]),\n                         max_err_m=float(e.max()), med_err_m=float(np.median(e)), w10=float((e <= 10).mean()),\n                         w20=float((e <= 20).mean()), w32=float((e <= 32).mean()),\n                         unknown=float(np.mean([s[\'unknown\'] for s in log]))))\n        if verbose and (i + 1) % 10 == 0:\n            print(f\'  {i + 1}/{len(segs)} 구간\', flush=True)\n    return pd.DataFrame(steps), pd.DataFrame(rows)\n\n\ndef summarize(steps, segs):\n    """한 줄 요약 dict. 걸음 비율은 출발 걸음(k = 0, 위치를 줌)을 뺀다."""\n    s = steps[steps.k != steps.groupby(\'set\').k.transform(\'min\')]\n    return dict(segments=len(segs), steps=len(s), w10=round(float((s.err_m <= 10).mean() * 100), 1),\n                w20=round(float((s.err_m <= 20).mean() * 100), 1), w32=round(float((s.err_m <= 32).mean() * 100), 1),\n                med_err_m=round(float(s.err_m.median()), 1), final_med_m=round(float(segs.final_err_m.median()), 1),\n                final_le40=round(float((segs.final_err_m <= 40).mean() * 100), 1),\n                unknown=round(float(s.unknown.mean() * 100), 1), outside_block=round(float(s.outside_block.mean() * 100), 1))\n', 'imatch/__init__.py': '"""Trone 이미지 매칭 파이프라인: 드론 영상 + 지도 블록 → 위치·heading (학습 없음).\n\n  geometry.py  ④⑤ MAGSAC++ 검증, 호모그래피 → 위치·heading\n  matchers.py  ②③ 특징 추출·매칭 (LoMa, RoMa v2, SIFT ...)\n  pipeline.py  ①~⑤ 한 샘플 처리 (회전 탐색 포함)\n  blocks.py    ⑥ 후보 블록 자르기, 이웃 블록 좌표\n  nav.py       1 km 닫힌 루프 항법 (2D 크롭, 3D 렌더 입력)\n  metrics.py   팀 공통 평가 규칙\n"""\ndef __getattr__(name):\n    # CPU evaluation should not require GPU/image-processing dependencies.\n    if name == \'get_matcher\':\n        from .matchers import get_matcher\n        return get_matcher\n    if name == \'localize\':\n        from .pipeline import localize\n        return localize\n    raise AttributeError(name)\n', 'evaluate.py': '\nimport sys,time,random,json\nfrom pathlib import Path\nimport torch, numpy as np, cv2\nfrom imatch.replay import run_segments,summarize\nfrom imatch.matchers import get_matcher\nfrom imatch.nav import MatchLoc\nfrom map_cache import StepMapMatcher\nname,data,destination=sys.argv[1:]\nout=Path(destination)\ndef seed():\n    random.seed(2026)\n    np.random.seed(2026)\n    torch.manual_seed(2026)\n    cv2.setRNGSeed(2026)\nseed()\nangles = [0, 90, 180, 270]\nmatcher=get_matcher(\'loma\',\'cuda\')\n# Both workers initialize the matcher using the same real image pair before timing.\nimport pandas as pd\nr=pd.read_csv(Path(data)/\'samples.csv\').sort_values([\'set\',\'k\']).iloc[0]\ndef rgb(path):\n    return cv2.cvtColor(cv2.imread(str(path)),cv2.COLOR_BGR2RGB)\nsample=rgb(Path(data)/r[\'set\']/f"k{int(r.k):03d}_uav.jpg")\nblock=rgb(Path(data)/\'maps\'/(r[\'set\']+\'.jpg\'))[:512,:512].copy()\nwith torch.inference_mode():\n    matcher(sample,block)\n    torch.cuda.synchronize()\n    seed()\n    matcher = StepMapMatcher(matcher, reuse=(name == \'map_cache\'))\n    cls = MatchLoc\n    start=time.perf_counter()\n    steps,segs=run_segments(lambda city: cls(matcher,city,angles=angles,stop_inl=15),\n                            data,odo=\'vio\',seed=2026,reject_inl=15)\n    torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\nmeasured=steps[steps.k!=steps.groupby(\'set\').k.transform(\'min\')]\nassert measured.n_ang.between(0, len(angles)).all()\nassert len(measured)==1020 and len(segs)==92\nsteps.to_csv(out/(name+\'_steps.csv\'),index=False)\nsegs.to_csv(out/(name+\'_segments.csv\'),index=False)\nrow=dict(variant=name, angles_deg=angles, map_extractions=matcher.extractions, cache_hits=matcher.hits,**summarize(steps,segs),wall_seconds=elapsed,\n         seconds_per_step=elapsed/len(steps),mean_angles=measured.n_ang.mean())\n(out/(name+\'_summary.json\')).write_text(json.dumps(row,indent=2))\nprint(row,flush=True)\n', 'map_cache.py': 'class StepMapMatcher:\n    """Instrument both variants. Reuse only the second image of the pinned LoMa pair API."""\n    def __init__(self, matcher, reuse):\n        self.matcher = matcher\n        self.reuse = reuse\n        self.block = self.features = None\n        self.hits = self.extractions = 0\n        model = matcher.m.matcher\n        self.extract = model.detect_and_describe\n        model.detect_and_describe = self.describe\n\n    def __call__(self, uav, block):\n        if block is not self.block:\n            self.block = block\n            self.features = None\n        self.image_index = 0\n        result = self.matcher(uav, block)\n        if self.image_index != 2:\n            raise RuntimeError(\'Expected UAV and map extraction once per LoMa pair\')\n        return result\n\n    def describe(self, image, count):\n        self.image_index += 1\n        if self.image_index == 1:\n            return self.extract(image, count)\n        if self.image_index != 2:\n            raise RuntimeError(\'Unexpected feature extraction order\')\n        if self.reuse and self.features is not None:\n            self.hits += 1\n            return self.features\n        self.extractions += 1\n        result = self.extract(image, count)\n        if self.reuse:\n            self.features = (result[0], result[1], None, None)\n        return result\n'}
for name,content in sources.items():
    path=SOURCE/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf8')
def evaluate(variant):
    subprocess.run([str(PY),'-u',str(SOURCE/'evaluate.py'),variant,str(DATA),str(RUN)],check=True)

In [ ]:
# 원본 4방향 R1
evaluate('r1')

In [ ]:
# 지도 특징 재사용 4방향 R1
evaluate('map_cache')

In [ ]:
variants = ['r1', 'map_cache']
comparison = pd.DataFrame([json.loads((RUN/f'{v}_summary.json').read_text()) for v in variants])
comparison['speedup_vs_r1'] = comparison.wall_seconds.iloc[0] / comparison.wall_seconds
comparison['seconds_saved'] = comparison.wall_seconds.iloc[0] - comparison.wall_seconds
comparison.rename(columns={'seconds_per_step':'seconds_per_frame'}, inplace=True)
a,b = [pd.read_csv(RUN/f'{v}_steps.csv').set_index(['set','k'], verify_integrity=True) for v in variants]
assert a.index.equals(b.index) and len(a)==1112
changes = pd.DataFrame(index=a.index)
changes['position_delta_m'] = np.hypot(a.est_x-b.est_x, a.est_y-b.est_y) / 4.0
for col in ['block_x','block_y','n_ang','n_inl','found','unknown','used']:
    changes[col+'_changed'] = a[col].ne(b[col])
assert np.isfinite(changes.position_delta_m).all()
changes.to_csv(RUN/'step_differences.csv')
equivalent = (changes.position_delta_m <= 1e-5) & ~changes.iloc[:,1:].any(axis=1)
print(f'동일 판정 및 위치 차이 0.00001m 이내: {equivalent.sum()} / {len(a)}, 최대 위치 차이: {changes.position_delta_m.max():.6f} m')
print('초기 프레임을 포함한 실제 지도 특징 추출 횟수와 재사용 횟수')
comparison.to_csv(RUN/'comparison.csv',index=False)
display(comparison[['variant','wall_seconds','seconds_saved','speedup_vs_r1','map_extractions','cache_hits','final_med_m','final_le40']])
archive=shutil.make_archive(str(RUN)+'_results','zip',RUN)
display(FileLink(os.path.relpath(archive)))
